In [ ]:
from glob import glob
from rich.pretty import pprint
from pydantic_ai import BinaryContent

from common.cache import RedisCache
from istm.llm_agents import ImageDescriber, ImageDescriberInput

In [ ]:
def get_binary_content(image_path: str) -> BinaryContent:
    with open(image_path, "rb") as image_file:
        return BinaryContent(
            data=image_file.read(),
            media_type="image/jpg",
        )

In [ ]:
image_paths = glob("/resources/cropped-train-images/*.jpg")
print(f"image_paths => {len(image_paths)}")

In [ ]:
description_guidelines = (
    "Analyze the provided image and generate a highly detailed description."
)

agent_inputs = [
    ImageDescriberInput(description_guidelines=description_guidelines)
    for _ in image_paths
]

user_contents = [
    get_binary_content(image_path=image_path) for image_path in image_paths
]

image_describer = ImageDescriber(cache=RedisCache(), max_concurrency=5)
image_describer_outputs = await image_describer.batch_generate(
    agent_inputs=agent_inputs,
    user_contents=user_contents,
)